# API Documentation Agent

This notebook walks through the full pipeline: it downloads and chunks an OpenAPI specification, ingests those chunks into a Vertex AI Search data store, and launches an interactive Gradio chat interface where you can ask natural-language questions about the API and receive answers grounded in the official documentation.

The Kubernetes API is used as the default example, but any OpenAPI 2.0/3.0 spec, Postman collection, or Markdown documentation works.

In [ ]:
!pip install -q uv
!uv pip install -q --system gradio google-api-core google-cloud-discoveryengine google-cloud-aiplatform==1.71.1 vertexai requests pyyaml 'mcp[cli]'

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
GCP_PROJECT_ID = "your-project-id"                # @param {type:"string"}
GCP_LOCATION = "global"                            # @param {type:"string"}
GEMINI_LOCATION = "us-central1"                   # @param {type:"string"}
VERTEX_SEARCH_DATA_STORE_ID = "your-engine-id"    # @param {type:"string"}
API_SPEC_URL = "https://raw.githubusercontent.com/kubernetes/kubernetes/v1.36.0/api/openapi-spec/swagger.json"  # @param {type:"string"}
API_NAME = "Kubernetes"                            # @param {type:"string"}
API_VERSION = "v1.36.0"                            # @param {type:"string"}

import os
os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
os.environ["GCP_LOCATION"] = GCP_LOCATION
os.environ["GEMINI_LOCATION"] = GEMINI_LOCATION
os.environ["VERTEX_SEARCH_DATA_STORE_ID"] = VERTEX_SEARCH_DATA_STORE_ID
os.environ["API_NAME"] = API_NAME

## Step 1: Ingest API Documentation

Parses the OpenAPI spec and writes one chunk per endpoint and one per schema to `data/<name>_<version>_chunks.jsonl`.

In [ ]:
!python -m src.ingest --spec {API_SPEC_URL} --name {API_NAME.lower()} --version {API_VERSION}

## Step 2: Upload to Vertex AI Search

1. Go to the [Vertex AI Search console](https://console.cloud.google.com/gen-app-builder/data-stores) and create a data store: **Structured Data → JSONL with document IDs**. Import `data/<name>_<version>_chunks.jsonl` and wait for indexing.
2. Go to [Engines](https://console.cloud.google.com/gen-app-builder/engines) and create a **Custom search (general)** app attached to that data store.
3. Copy the **Engine ID** and paste it into `VERTEX_SEARCH_DATA_STORE_ID` in the config cell above, then re-run that cell.

## Step 3: Launch the Documentation Agent

In [ ]:
import os
os.chdir("/content/api-rag")  # adjust if repo is cloned elsewhere

from src.app import demo
demo.launch(share=True)